# Day 2.1 — Documents and Chunks

The model does not automatically know our fictional campus documents. First we make the documents searchable.

```text
Markdown files → sections → chunks with metadata
```

A chunk is a retrieval unit, not an arbitrary character slice.

## Before you begin

### Learning outcomes

Inspect supplied documents and create chunks that retain source and section metadata.

Architecture reference: [D06](../../diagrams/source/day_02.md).

### Expected observation

Every chunk has stable text, source, section, and identifier fields.


## Concept briefing

## Why retrieval is an application problem

A model may know general facts, but a course application often needs supplied manuals,
project documents or current organisational information. Placing every document in every
request is expensive, noisy and eventually impossible. Retrieval selects a small amount
of evidence relevant to the current question and places it into the model context.

Retrieval-Augmented Generation is therefore a pipeline, not a model feature:

```text
documents -> chunks -> representations -> index
question -> retrieval -> selected evidence -> generation -> validation
```

Every arrow can fail. Debugging RAG requires identifying which arrow failed rather than
changing prompts at random.

## Why documents become chunks

Retrieval operates on units. A whole manual may contain the answer but also thousands of
irrelevant words. A tiny fragment may match a keyword but lack the surrounding condition
that changes its meaning. Chunking balances retrieval precision against sufficient
context.

Useful chunks retain provenance: source file, section heading, stable identifier and
text. Without this metadata the application cannot cite the result, evaluate expected
sections, or explain why a passage was retrieved.

There is no universal chunk size. Structure-aware chunks are often easier to inspect than
blind character windows for small engineering documents. The course therefore starts
with headings rather than presenting chunking as an arbitrary numeric tuning exercise.


In [ ]:
import sys
from pathlib import Path
here = Path.cwd().resolve()
candidates = [here, here / "day_02_knowledge_and_state", here.parent]
project_root = next(p for p in candidates if (p / "src" / "knowledge_agent").exists())
sys.path.insert(0, str(project_root / "src"))
from knowledge_agent.documents import load_markdown_corpus
corpus_dir = project_root / "data" / "corpus"

## Inspect the source before processing

The corpus contains three small fictional engineering documents. Keeping it small lets us inspect every retrieval failure.

In [ ]:
for path in sorted(corpus_dir.glob("*.md")):
    print(path.name, path.stat().st_size, "bytes")

In [ ]:
chunks = load_markdown_corpus(corpus_dir)
print("chunks:", len(chunks))
for chunk in chunks[:4]:
    print("\n", chunk.chunk_id, "|", chunk.source, "|", chunk.section)
    print(chunk.text[:180])

## Observe

Each chunk preserves source, document title, section, ID, and text. Metadata later supports citations and filtering. If we discard it during ingestion, the model cannot recreate trustworthy provenance.

## Exercise

Find the chunk containing the five-minute reconnection rule. Print its ID, source, section, and full text. Then explain why one-section-per-chunk is reasonable for this corpus and when it might fail.

## Checkpoint

We transformed documents into identifiable retrieval units. We have not used embeddings, a vector database, or a model yet. Next we establish a simple keyword-search baseline.

## Your turn

Change chunk size or heading boundaries and compare one resulting record.

## Recap

Retrieval quality depends on the units indexed, not only the model.
